# EDA — daily online retail sales

Exploratory analysis of ~2 years of daily sales revenue from an online store
(UCI *Online Retail II*, aggregated to one revenue total per day by
`backend/data/prepare_dataset.py`).

We look at trend, weekly seasonality, closed days, the value distribution,
outliers and autocorrelation, then decide what cleaning the series needs before
forecasting. Every section prints its numbers; the charts render wherever
matplotlib is installed and save into `docs/`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.seasonal import STL

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="talk")
    HAS_MPL = True
except Exception as e:
    HAS_MPL = False
    print("matplotlib not available here, charts skipped:", e)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOCS = REPO / "docs"; DOCS.mkdir(exist_ok=True)

df = pd.read_csv(REPO / "backend/data/online_retail_daily.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
df.head()

matplotlib not available here, charts skipped: DLL load failed while importing _image: An Application Control policy has blocked this file.


,date,value
0,2009-12-01,54513.50
1,2009-12-02,63352.51
2,2009-12-03,74037.91
3,2009-12-04,40732.92
4,2009-12-05,9803.05


## 1. Overview

In [2]:
span = (df.date.max() - df.date.min()).days + 1
print("rows            :", len(df))
print("date range      :", df.date.min().date(), "->", df.date.max().date())
print("calendar days   :", span, "| missing dates:", span - len(df))
print("zero-sales days :", int((df.value == 0).sum()))
df.value.describe().round(1)

rows            : 739
date range      : 2009-12-01 -> 2011-12-09
calendar days   : 739 | missing dates: 0
zero-sales days : 135


count       739.0
mean      28380.2
std       23366.3
min           0.0
25%       14793.1
50%       25955.0
75%       38033.1
max      200938.6
Name: value, dtype: float64

## 2. The series over time\nDaily revenue with a 30-day moving average to expose the trend.

In [3]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(13, 4.5))
    ax.plot(df.date, df.value, lw=0.8, color="#4c78a8", label="daily sales")
    ax.plot(df.date, df.value.rolling(30).mean(), lw=2.2, color="#e45756", label="30-day average")
    ax.set(title="Daily online retail sales (revenue)", ylabel="revenue (GBP)")
    ax.legend(); fig.tight_layout(); fig.savefig(DOCS / "eda_series.png", dpi=110); plt.show()
print("first vs last 30-day average: %.0f -> %.0f"
      % (df.value.rolling(30).mean().iloc[29], df.value.rolling(30).mean().iloc[-1]))

first vs last 30-day average: 27523 -> 56469


## 3. Weekly seasonality\nAverage sales by day of week.

In [4]:
df["weekday"] = df.date.dt.day_name()
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
by_wd = df.groupby("weekday")["value"].mean().reindex(order).round(0)
print(by_wd.to_string())
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(x=by_wd.index, y=by_wd.values, color="#4c78a8", ax=ax)
    ax.set(title="Average sales by weekday", ylabel="avg revenue (GBP)")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout(); fig.savefig(DOCS / "eda_weekday.png", dpi=110); plt.show()

weekday
Monday       34716.0
Tuesday      39489.0
Wednesday    33979.0
Thursday     40626.0
Friday       31858.0
Saturday        93.0
Sunday       17591.0


## 4. Closed days\nWhere do the zero-sales days fall?

In [5]:
zero = df[df.value == 0]
print("zero-sales days:", len(zero))
print(zero.date.dt.day_name().value_counts().to_string())

zero-sales days: 135
date
Saturday     104
Monday        11
Friday         7
Sunday         6
Thursday       3
Tuesday        2
Wednesday      2


## 5. Value distribution\nRaw revenue is right-skewed; a log transform makes it roughly symmetric.

In [6]:
print("skew raw   :", round(df.value.skew(), 2))
print("skew log1p :", round(np.log1p(df.value).skew(), 2))
if HAS_MPL:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    sns.histplot(df.value, bins=40, color="#4c78a8", ax=ax[0]); ax[0].set(title="Sales (raw)")
    sns.histplot(np.log1p(df.value), bins=40, color="#54a24b", ax=ax[1]); ax[1].set(title="log(1 + sales)")
    fig.tight_layout(); fig.savefig(DOCS / "eda_distribution.png", dpi=110); plt.show()

skew raw   : 1.81
skew log1p : -1.58


## 6. Outliers\nThe biggest days are real pre-Christmas surges, not errors.

In [7]:
print("largest days:")
print(df.nlargest(5, "value")[["date", "value"]].to_string(index=False))
q1, q3 = df.value.quantile([.25, .75]); iqr = q3 - q1
fence = q3 + 1.5 * iqr
print(f"\nIQR upper fence: {fence:,.0f}  | days above it: {int((df.value > fence).sum())}")

largest days:
      date     value
2011-12-09 200938.60
2010-12-07 199236.40
2010-09-27 118909.67
2010-12-01 117921.58
2011-11-14 114419.89

IQR upper fence: 72,893  | days above it: 30


## 7. Autocorrelation\nHow strongly does the past predict the future? Watch lag 7.

In [8]:
a = acf(df.value, nlags=35)
for lag in (1, 7, 14, 21, 28):
    print(f"acf(lag={lag:>2}) = {a[lag]:.3f}")
if HAS_MPL:
    from statsmodels.graphics.tsaplots import plot_acf
    fig, ax = plt.subplots(figsize=(11, 4))
    plot_acf(df.value, lags=35, ax=ax); ax.set_title("Autocorrelation (lag in days)")
    fig.tight_layout(); fig.savefig(DOCS / "eda_acf.png", dpi=110); plt.show()

acf(lag= 1) = 0.442
acf(lag= 7) = 0.575
acf(lag=14) = 0.464
acf(lag=21) = 0.419
acf(lag=28) = 0.414


## 8. STL decomposition\nSplit into trend + weekly seasonality + remainder, and measure the strength of each.

In [9]:
s = df.set_index("date")["value"].asfreq("D")
res = STL(s, period=7, robust=True).fit()
rv = res.resid.var()
Ft = max(0, 1 - rv / (res.trend + res.resid).var())
Fs = max(0, 1 - rv / (res.seasonal + res.resid).var())
print(f"trend strength    : {Ft:.2f}")
print(f"seasonal strength : {Fs:.2f}   (0 = none, 1 = dominant)")
if HAS_MPL:
    fig = res.plot(); fig.set_size_inches(12, 7); fig.tight_layout()
    fig.savefig(DOCS / "eda_decomposition.png", dpi=110); plt.show()

trend strength    : 0.50
seasonal strength : 0.56   (0 = none, 1 = dominant)


## Takeaways for modelling

- **Continuous daily calendar, no missing dates** — every day is present after aggregation.
- **Strong weekly seasonality** — the store is effectively closed on Saturdays (near-zero), and
  the ACF spikes at lag 7. XGBoost gets a `dayofweek` feature; Chronos picks the weekly pattern
  up from context.
- **Upward trend + a large pre-Christmas surge** — the biggest days are real, so we keep them
  rather than clipping.
- **Right-skewed values with hard zeros** — MAE / RMSE are the reliable scores here; MAPE is
  unstable because of the zero-sales days.
- **Cleaning stays light**: keep the real zeros and spikes, just guarantee the continuous daily
  index the models expect.